In [0]:
from pyspark import SparkFiles
p1 ="https://github.com/erkansirin78/datasets/raw/master/churn-telecom/cell2celltrain_clean_parquet/part-00000-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet"
p2 ="https://github.com/erkansirin78/datasets/raw/master/churn-telecom/cell2celltrain_clean_parquet/part-00001-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet"
p3 ="https://github.com/erkansirin78/datasets/raw/master/churn-telecom/cell2celltrain_clean_parquet/part-00002-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet"
p4 ="https://github.com/erkansirin78/datasets/raw/master/churn-telecom/cell2celltrain_clean_parquet/part-00003-b4b53465-f422-483e-b076-772ed0ed5b76-c000.snappy.parquet"
sc.addFile(p1)
sc.addFile(p2)
sc.addFile(p3)
sc.addFile(p4)
df = spark.read.parquet("file:///"+SparkFiles.get("*"))

In [0]:
df.count()

Out[10]: 51046

In [0]:
# column seperation after analysis
categoric_cols = []
numeric_cols = []
# Why we discard these? Because they have high cardinality. 
# In other words they have too many distinct categories
# Adding them to model will not do good if not too bad.
discarted_cols = ['CustomerID', 'ServiceArea']
label_col = ['Churn']

In [0]:
for col_name in df.dtypes:
    if (col_name[0] not in discarted_cols+label_col):
        if (col_name[1] == 'string'):
            categoric_cols.append(col_name[0])
        else: numeric_cols.append(col_name[0])

In [0]:
# Select columns for onehotencoder
to_be_onehotencoded_cols = []

for col_name in categoric_cols:
    count = df.select(col_name).distinct().count()
    if count > 2:
        to_be_onehotencoded_cols.append(col_name)
    print("{} has {} distinct category.".format(col_name, count))

ChildrenInHH has 2 distinct category.
HandsetRefurbished has 2 distinct category.
HandsetWebCapable has 2 distinct category.
TruckOwner has 2 distinct category.
RVOwner has 2 distinct category.
Homeownership has 2 distinct category.
BuysViaMailOrder has 2 distinct category.
RespondsToMailOffers has 2 distinct category.
OptOutMailings has 2 distinct category.
NonUSTravel has 2 distinct category.
OwnsComputer has 2 distinct category.
HasCreditCard has 2 distinct category.
NewCellphoneUser has 2 distinct category.
NotNewCellphoneUser has 2 distinct category.
OwnsMotorcycle has 2 distinct category.
MadeCallToRetentionTeam has 2 distinct category.
CreditRating has 7 distinct category.
PrizmCode has 4 distinct category.
Occupation has 8 distinct category.
MaritalStatus has 3 distinct category.


In [0]:
print(to_be_onehotencoded_cols)

['CreditRating', 'PrizmCode', 'Occupation', 'MaritalStatus']


In [0]:
# StringIndexer
from pyspark.ml.feature import StringIndexer

# Will hold stringIndexer objects and column names
my_dict = {}

# Will collect StringIndexer ojects
string_indexer_objs = []

# Will collect StringIndexer output colnames
string_indexer_output_names = []

# Will collect OneHotEncoder output colnames
ohe_col_input_names = []
ohe_col_output_names = []

for col_name in categoric_cols:
    my_dict[col_name+"_indexobj"] = StringIndexer() \
    .setHandleInvalid('skip') \
    .setInputCol(col_name) \
    .setOutputCol(col_name+"_indexed")
    
    string_indexer_objs.append(my_dict.get(col_name+"_indexobj"))
    
    string_indexer_output_names.append(col_name+"_indexed")
    if col_name in to_be_onehotencoded_cols:
        ohe_col_input_names.append(col_name+"_indexed")
        ohe_col_output_names.append(col_name+"_ohe")

In [0]:
# OnehotEncoder

from pyspark.ml.feature import OneHotEncoder

encoder = OneHotEncoder() \
.setInputCols(ohe_col_input_names) \
.setOutputCols(ohe_col_output_names)

In [0]:
# VectorAssembler
from pyspark.ml.feature import VectorAssembler

# We have to exclude onehotencoded categoric cols from stringIndexer cols
string_indexer_col_names_ohe_exluded = list(set(string_indexer_output_names).difference(set(ohe_col_input_names)))

assembler = VectorAssembler().setHandleInvalid("skip") \
.setInputCols(numeric_cols+string_indexer_col_names_ohe_exluded+ohe_col_output_names) \
.setOutputCol('unscaled_features')

In [0]:
# LabelIndexer

label_indexer = StringIndexer().setHandleInvalid("skip") \
.setInputCol(label_col[0]) \
.setOutputCol("label")

In [0]:
# Feature Scaling

from pyspark.ml.feature import StandardScaler
scaler = StandardScaler().setInputCol("unscaled_features").setOutputCol("scaled_features")

In [0]:
# PCA
from pyspark.ml.feature import PCA
pca = PCA().setInputCol("scaled_features").setK(10).setOutputCol("features")

In [0]:
# Estimator: GBTClassifier
from pyspark.ml.classification import GBTClassifier

estimator = GBTClassifier() \
.setFeaturesCol("features") \
.setLabelCol("label")

In [0]:
# Pipeline
from pyspark.ml import Pipeline

pipeline_obj = Pipeline().setStages(string_indexer_objs+[encoder, assembler, label_indexer, scaler, pca, estimator])

In [0]:
# Split data
train_df, test_df = df.randomSplit([0.8,0.2], seed=142)

In [0]:
print(train_df.count())
print(test_df.count())

40699
10347


In [0]:
# Train Model
pipeline_model = pipeline_obj.fit(train_df)

In [0]:
# Save the model

pipeline_model.write().overwrite().save("saved_models/gbt_churn_pca10")

In [0]:
transformed_df = pipeline_model.transform(test_df)

In [0]:
transformed_df.select('label','prediction','probability','rawPrediction').show(truncate=False)

+-----+----------+----------------------------------------+------------------------------------------+
|label|prediction|probability                             |rawPrediction                             |
+-----+----------+----------------------------------------+------------------------------------------+
|0.0  |0.0       |[0.7828692358531142,0.2171307641468858] |[0.64123295310985,-0.64123295310985]      |
|0.0  |0.0       |[0.7106893193993361,0.2893106806006639] |[0.44936712068233364,-0.44936712068233364]|
|0.0  |0.0       |[0.6545586040235132,0.3454413959764868] |[0.3195690567856497,-0.3195690567856497]  |
|0.0  |0.0       |[0.8380295739488679,0.1619704260511321] |[0.8218198138202891,-0.8218198138202891]  |
|0.0  |0.0       |[0.665115923097792,0.334884076902208]   |[0.34308845653489095,-0.34308845653489095]|
|1.0  |0.0       |[0.6309729668979359,0.3690270331020641] |[0.2681965590720137,-0.2681965590720137]  |
|0.0  |0.0       |[0.8210911661628462,0.17890883383715384]|[0.76187888909

In [0]:
# Evaluate the model
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator()

In [0]:
evaluator.evaluate(transformed_df)

Out[41]: 0.5887646774920137

In [0]:
evaluator.getMetricName()

Out[42]: 'areaUnderROC'

In [0]:
# Get churn score from probability
import pyspark.sql.functions as F
import pyspark.sql.types as T

to_array = F.udf(lambda v: v.toArray().tolist(), T.ArrayType(T.FloatType()))
score_df = transformed_df.select('CustomerID','label','prediction','probability','rawPrediction') \
.withColumn('score', to_array('probability')[1] * 1000)

In [0]:
score_df.filter("prediction = 1.0").orderBy(F.desc("score")).show(truncate=False)

+----------+-----+----------+----------------------------------------+------------------------------------------+---------+
|CustomerID|label|prediction|probability                             |rawPrediction                             |score    |
+----------+-----+----------+----------------------------------------+------------------------------------------+---------+
|3168050   |0.0  |1.0       |[0.22368778132129832,0.7763122186787017]|[-0.6221517683720942,0.6221517683720942]  |776.31226|
|3320426   |0.0  |1.0       |[0.2274567110617437,0.7725432889382563] |[-0.6113640527121987,0.6113640527121987]  |772.54333|
|3085130   |1.0  |1.0       |[0.22790085929967374,0.7720991407003263]|[-0.6101011278096765,0.6101011278096765]  |772.0991 |
|3319214   |1.0  |1.0       |[0.23942137560282623,0.7605786243971737]|[-0.577927207061419,0.577927207061419]    |760.5786 |
|3153570   |1.0  |1.0       |[0.27644538012789843,0.7235546198721016]|[-0.4810813879734317,0.4810813879734317]  |723.5546 |
|3078238

In [0]:
# Hyperparameter Tuning
from pyspark.ml.tuning import CrossValidatorModel, ParamGridBuilder, CrossValidator

paramGrid = ParamGridBuilder() \
.addGrid(pca.k, [15, 25]) \
.addGrid(estimator.maxDepth, [5, 10]) \
.addGrid(estimator.maxBins, [32, 50]) \
.build()

In [0]:
crossval = CrossValidator(estimator=pipeline_obj,
                          estimatorParamMaps=paramGrid,
                          evaluator=BinaryClassificationEvaluator(),
                          numFolds=5) 

In [0]:
import time
start_time = time.time()
cvModel = crossval.fit(df)

print("----------- %s secs ----------" %(time.time() - start_time))

In [0]:
evaluator.evaluate(cvModel.transform(df))